In [ ]:
# Cell 1: Install (restart runtime after this)
!pip install --upgrade transformers peft bitsandbytes accelerate numpy==1.26.4 fastapi uvicorn pyngrok

In [ ]:
# Cell 2: Load model from Drive
from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
login(token="YOUR_HUGGINGFACE_TOKEN")

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, "PATH_TO_ADAPTER")
tokenizer = AutoTokenizer.from_pretrained("PATH_TO_ADAPTER")
tokenizer.pad_token = tokenizer.eos_token
model.eval()
print("Model loaded!")

In [ ]:
# Cell 3: Start API server with ngrok


# Write the API server file

!ngrok authtoken YOUR_NGROK_TOKEN


api_code = '''
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, List
import torch
import time

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    message: str
    history: Optional[List[dict]] = None
    max_tokens: Optional[int] = 256
    temperature: Optional[float] = 0.7

class ChatResponse(BaseModel):
    response: str
    processing_time_ms: float

SYSTEM_PROMPT = """You are a helpful customer support assistant. You provide clear,
accurate, and professional responses to customer inquiries. Be concise but thorough,
and always maintain a polite and supportive tone."""

model = None
tokenizer = None

@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")

    start_time = time.time()

    if request.history:
        prompt = f"<s>[INST] {SYSTEM_PROMPT}\\n\\n"
        for msg in request.history:
            if msg["role"] == "user":
                prompt += f"Customer: {msg[\"content\"]} [/INST] "
            elif msg["role"] == "assistant":
                prompt += f"{msg[\"content\"]}</s><s>[INST] "
        prompt += f"Customer: {request.message} [/INST]"
    else:
        prompt = f"<s>[INST] {SYSTEM_PROMPT}\\n\\nCustomer: {request.message} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_tokens,
            temperature=request.temperature,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    elapsed = (time.time() - start_time) * 1000
    return ChatResponse(response=response_text, processing_time_ms=round(elapsed, 2))

@app.get("/health")
async def health():
    return {"status": "healthy", "model_loaded": model is not None}
'''

with open('demo_api.py', 'w') as f:
    f.write(api_code)

# Import and inject the model
import demo_api
demo_api.model = model
demo_api.tokenizer = tokenizer

# Start server in background
import threading
import uvicorn

def run_server():
    uvicorn.run(demo_api.app, host="0.0.0.0", port=8000, log_level="info")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Create ngrok tunnel
import time
time.sleep(2)

from pyngrok import ngrok
public_url = ngrok.connect(8000)

print("\n" + "=" * 60)
print("YOUR API IS LIVE!")
print("=" * 60)
print(f"\nPublic URL: {public_url}")
print(f"\nPaste this URL into your HTML chat file and click Connect")
print(f"\nTest it: {public_url}/health")
print("=" * 60)